
# CMR Client — Quickstart Tests

This notebook helps you **exercise** `dswxni/cmr_client.py` with real CMR queries in a controlled way.

**What you'll test:**

1. Import the module
2. Search for granules (ASF ALOS‑1 HRTC now; NISAR GCOV later)
3. Inspect results and pick a download URL
4. (Optional) Download a file to local cache

> ⚠️ You need an Earthdata Login. For fastest auth, create a `~/.netrc` entry for `urs.earthdata.nasa.gov`:
>
> ```
> machine urs.earthdata.nasa.gov login <USER> password <PASS>
> ```
>
> Alternatively, export `DSWXNI_EARTHDATA_USERNAME` and `DSWXNI_EARTHDATA_PASSWORD`.


## 0) Environment & install (run in repo root)

In [1]:

# (Run this only once, in your local environment)
# Install the project in editable mode so the notebook imports your local code
# !python -m venv .venv && source .venv/bin/activate
# !pip install -e .
# Optionally: set provider defaults for the session
import os
os.environ.setdefault("DSWXNI_PARALLEL_DOWNLOADS", "6")


'6'

## 1) Import the CMR client

In [2]:

from src.cmr_client import (
    search_cmr_flexible, search_cmr,
    CollectionSpec, Spatial, Temporal, ExtraFilters
)
from pprint import pprint



## 2) Example ALOS‑1 HRTC (ASF)

> Replace `short_name` or use a known `concept_id` once you confirm the exact collection in CMR.
> If you're unsure, start with `short_name` like `"ALOS_PALSAR_RTC_HiRes"` (example) and adjust.


In [6]:

spec_asf = CollectionSpec(
    concept_id="C1206487504-ASF",   # ALOS-1 High Res Terrain Corrected (ASF)
    provider="ASF",
    version=None  # e.g., "003" if applicable
)

spatial = Spatial(bbox=(-124, 45, -121.5, 46.5))  # Oregon-ish bbox; adjust
temporal = Temporal(start="2007-01-01T00:00:00Z", end="2011-12-31T23:59:59Z")

granules = search_cmr_flexible(spec_asf, spatial=spatial, temporal=temporal, max_items=20)
print(f"Found {len(granules)} granules")
for g in granules[:]:
    print("—", g.title, "| provider:", g.provider, "| size(MB):", g.size_mb)


Found 20 granules
— ALPSRP050500890-RTC_HI_RES | provider: ASF | size(MB): 244.47
— ALPSRP050500900-RTC_HI_RES | provider: ASF | size(MB): 235.54
— ALPSRP050500910-RTC_HI_RES | provider: ASF | size(MB): 240.28
— ALPSRP050500920-RTC_HI_RES | provider: ASF | size(MB): 238.28
— ALPSRP052250890-RTC_HI_RES | provider: ASF | size(MB): 232.17
— ALPSRP052250900-RTC_HI_RES | provider: ASF | size(MB): 237.21
— ALPSRP052250910-RTC_HI_RES | provider: ASF | size(MB): 236.77
— ALPSRP052980890-RTC_HI_RES | provider: ASF | size(MB): 228.92
— ALPSRP052980900-RTC_HI_RES | provider: ASF | size(MB): 226.7
— ALPSRP052980910-RTC_HI_RES | provider: ASF | size(MB): 233.94
— ALPSRP052980920-RTC_HI_RES | provider: ASF | size(MB): 233.8
— ALPSRP057210890-RTC_HI_RES | provider: ASF | size(MB): 244.74
— ALPSRP057210900-RTC_HI_RES | provider: ASF | size(MB): 235.57
— ALPSRP057210910-RTC_HI_RES | provider: ASF | size(MB): 240.97
— ALPSRP057210920-RTC_HI_RES | provider: ASF | size(MB): 239.31
— ALPSRP057940890-RTC_HI

### Pick a best download URL

In [4]:

if granules:
    g0 = granules[0]
    url = g0.pick_best_download(provider_hint="ASF")
    print("Chosen URL:", url)
else:
    print("No granules returned; adjust bbox/temporal/short_name")


Chosen URL: https://datapool.asf.alaska.edu/RTC_HI_RES/A3/AP_05050_FBS_F0890_RT1.zip


## 3) Backward-compatible wrapper test

In [7]:

granules_simple = search_cmr(
    "ALOS_PALSAR_RTC_HiRes",
    (-124, 45, -121.5, 46.5),
    "2007-01-01T00:00:00Z",
    "2011-12-31T23:59:59Z",
    max_items=100,
    provider="ASF",
)
print(f"Wrapper returned {len(granules_simple)} granules")
print(granules_simple[0].to_dict() if granules_simple else "None")

for g in granules_simple[:]:
    print("—", g.title, "| provider:", g.provider, "| size(MB):", g.size_mb)


Wrapper returned 100 granules
{'id': 'G1210891996-ASF', 'title': 'ALPSRP050500890-RTC_HI_RES', 'collection': 'ALOS_PALSAR_RTC_HIGH_RES', 'time_start': '2007-01-05T06:30:48.000Z', 'time_end': '2007-01-05T06:30:57.000Z', 'size_mb': 244.47, 'links': [{'href': 'https://datapool.asf.alaska.edu/RTC_HI_RES/A3/AP_05050_FBS_F0890_RT1.zip', 'rel': 'http://esipfed.org/ns/fedsearch/1.1/data#', 'title': None, 'type': None, 'inherited': None}, {'href': 'https://datapool.asf.alaska.edu/BROWSE/A3/ALPSRP050500890.jpg', 'rel': 'http://esipfed.org/ns/fedsearch/1.1/browse#', 'title': None, 'type': None, 'inherited': None}, {'href': 'https://datapool.asf.alaska.edu/BROWSE/A3/AP_05050_FBS_F0890.jpg', 'rel': 'http://esipfed.org/ns/fedsearch/1.1/browse#', 'title': None, 'type': None, 'inherited': None}, {'href': 'https://search.asf.alaska.edu/', 'rel': 'http://esipfed.org/ns/fedsearch/1.1/data#', 'title': None, 'type': None, 'inherited': True}, {'href': 'https://search.earthdata.nasa.gov/search/granules?p=C12


## 4) (Future) NISAR GCOV on PO.DAAC

This will work once the collection is public. Keep the call structure identical and update `short_name` or `concept_id`.

> Example only — will return zero until the collection exists.


In [10]:

spec_podaac = CollectionSpec(
    short_name="NISAR_L2_GCOV",  # placeholder; update when live
    provider="POCLOUD",
)
granules_gcov = search_cmr_flexible(
    spec_podaac,
    spatial=Spatial(bbox=(-122.6, 37.0, -121.5, 38.0)),
    temporal=Temporal(start="2026-01-01T00:00:00Z", end="2026-12-31T23:59:59Z"),
    max_items=5,
)
print("NISAR GCOV granules (expected 0 until live):", len(granules_gcov))


NISAR GCOV granules (expected 0 until live): 0



## 5) (Optional) Download a file

If you want to confirm your auth pipeline and storage paths, you can download the chosen URL.
This uses `requests` directly just for a quick smoke test; your package has a richer downloader.


In [11]:

# Quick ad-hoc download (for smoke testing only)
# Make sure you have Earthdata auth via ~/.netrc or env vars.
import requests, pathlib

if granules:
    url = granules[0].pick_best_download(provider_hint="ASF")
    if url:
        out = pathlib.Path("data_test"); out.mkdir(exist_ok=True, parents=True)
        dest = out / url.split("?")[0].split("/")[-1]
        with requests.get(url, stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(dest, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024*1024):
                    if chunk:
                        f.write(chunk)
        print("Downloaded to:", dest)
    else:
        print("No suitable URL found to download.")
else:
    print("No granules to download.")


Downloaded to: data_test/AP_05050_FBS_F0890_RT1.zip
